# Credit Card Data Quality Validation

This notebook validates the Bronze and Silver data produced by the
Credit Card Lakeflow Declarative Pipeline.

## Validation areas

1. Row counts
2. Primary key and duplicate checks
3. Referential integrity
4. NULL checks
5. Invalid values
6. Date and amount validation
7. Bronze vs Silver validation


## 1. ROW COUNT VALIDATION

In [0]:
%sql

SELECT
    'bronze.users' AS table_name,
    COUNT(*) AS row_count
FROM credit_card_project.bronze.users

UNION ALL

SELECT
    'bronze.cards' AS table_name,
    COUNT(*) AS row_count
FROM credit_card_project.bronze.cards

UNION ALL

SELECT
    'bronze.transactions' AS table_name,
    COUNT(*) AS row_count
FROM credit_card_project.bronze.transactions

UNION ALL

SELECT
    'silver.cleaned_transactions' AS table_name,
    COUNT(*) AS row_count
FROM credit_card_project.silver.cleaned_transactions;


Check 1 — Row count validation: PASS

## 2. Duplicate and Primary Key Validation

The purpose of this section is to verify that the expected key columns uniquely identify records in each dimension and transaction table.

In [0]:
%sql
-- ============================================================
-- 2A. USERS: DUPLICATE user_id CHECK
-- Expected result: 0
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS distinct_user_ids,
    COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_user_id_rows
FROM credit_card_project.bronze.users;


In [0]:
%sql
-- ============================================================
-- 2B. CARDS: DUPLICATE card_id CHECK
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT card_id) AS distinct_card_ids,
    COUNT(*) - COUNT(DISTINCT card_id) AS duplicate_card_id_rows
FROM credit_card_project.bronze.cards;


In [0]:
%sql
-- ============================================================
-- 2C. CARDS: COMPOSITE KEY CHECK
-- Expected result: 0 duplicate combinations
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(user_id, '-', card_id)) AS distinct_card_keys,
    COUNT(*) - COUNT(DISTINCT CONCAT(user_id, '-', card_id)) AS duplicate_card_key_rows
FROM credit_card_project.bronze.cards;


Therefore the appropriate key is: (user_id, card_id)

In [0]:
%sql
-- ============================================================
-- 2D. TRANSACTIONS: INSPECT SCHEMA
-- ============================================================

DESCRIBE credit_card_project.bronze.transactions;


In [0]:
%sql
-- ============================================================
-- 2E. TRANSACTIONS: EXACT DUPLICATE CHECK
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CONCAT_WS(
            '||',
            CAST(user_id AS STRING),
            CAST(card_id AS STRING),
            CAST(year AS STRING),
            CAST(month AS STRING),
            CAST(day AS STRING),
            time,
            amount,
            use_chip,
            CAST(merchant_name AS STRING),
            merchant_city,
            merchant_state,
            CAST(zip AS STRING),
            CAST(mcc AS STRING),
            errors,
            is_fraud
        )
    ) AS distinct_transaction_records,
    COUNT(*) - COUNT(DISTINCT
        CONCAT_WS(
            '||',
            CAST(user_id AS STRING),
            CAST(card_id AS STRING),
            CAST(year AS STRING),
            CAST(month AS STRING),
            CAST(day AS STRING),
            time,
            amount,
            use_chip,
            CAST(merchant_name AS STRING),
            merchant_city,
            merchant_state,
            CAST(zip AS STRING),
            CAST(mcc AS STRING),
            errors,
            is_fraud
        )
    ) AS exact_duplicate_rows
FROM credit_card_project.bronze.transactions;


In [0]:
%sql
-- ============================================================
-- 2F. INSPECT EXACT DUPLICATES
-- ============================================================

SELECT
    user_id,
    card_id,
    year,
    month,
    day,
    time,
    amount,
    use_chip,
    merchant_name,
    merchant_city,
    merchant_state,
    zip,
    mcc,
    errors,
    is_fraud,
    COUNT(*) AS duplicate_count
FROM credit_card_project.bronze.transactions
GROUP BY
    user_id,
    card_id,
    year,
    month,
    day,
    time,
    amount,
    use_chip,
    merchant_name,
    merchant_city,
    merchant_state,
    zip,
    mcc,
    errors,
    is_fraud
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;


 33 duplicated transaction combinations.

Check 2 — Duplicate Validation: FAILED

##3. Referential integrity

In [0]:
%sql
-- ============================================================
-- 3A. REFERENTIAL INTEGRITY: CARDS → USERS
-- Expected result: 0
-- ============================================================

SELECT COUNT(*) AS orphaned_cards
FROM credit_card_project.bronze.cards c
LEFT JOIN credit_card_project.bronze.users u
    ON c.user_id = u.user_id
WHERE u.user_id IS NULL;


In [0]:
%sql
-- ============================================================
-- 3B. REFERENTIAL INTEGRITY: TRANSACTIONS → USERS
-- Expected result: 0
-- ============================================================

SELECT COUNT(*) AS orphaned_transactions
FROM credit_card_project.bronze.transactions t
LEFT JOIN credit_card_project.bronze.users u
    ON t.user_id = u.user_id
WHERE u.user_id IS NULL;


In [0]:
%sql
-- ============================================================
-- 3C. REFERENTIAL INTEGRITY: TRANSACTIONS → CARDS
-- Expected result: 0
-- ============================================================

SELECT COUNT(*) AS orphaned_card_transactions
FROM credit_card_project.bronze.transactions t
LEFT JOIN credit_card_project.bronze.cards c
    ON t.user_id = c.user_id
   AND t.card_id = c.card_id
WHERE c.user_id IS NULL;


Check 3 — Referential integrity: PASS

##4. NULL Value Validation

This section identifies missing values in key business and relationship fields.

A NULL value is not automatically an error; some attributes may legitimately be missing. The checks below focus first on fields that are expected to be populated.

In [0]:
%sql
-- ============================================================
-- 4A. USERS: NULL CHECK
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(user_id) AS user_id_populated,
    COUNT(person) AS person_populated,
    COUNT(gender) AS gender_populated,
    COUNT(city) AS city_populated,
    COUNT(state) AS state_populated,
    COUNT(fico_score) AS fico_score_populated,
    COUNT(num_credit_cards) AS num_credit_cards_populated
FROM credit_card_project.bronze.users;


In [0]:
%sql
-- ============================================================
-- 4B. CARDS: NULL CHECK
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(user_id) AS user_id_populated,
    COUNT(card_id) AS card_id_populated,
    COUNT(card_brand) AS card_brand_populated,
    COUNT(card_type) AS card_type_populated,
    COUNT(expires) AS expires_populated,
    COUNT(has_chip) AS has_chip_populated,
    COUNT(cards_issued) AS cards_issued_populated,
    COUNT(credit_limit) AS credit_limit_populated,
    COUNT(acct_open_date) AS acct_open_date_populated,
    COUNT(year_pin_last_changed) AS year_pin_last_changed_populated,
    COUNT(card_on_dark_web) AS card_on_dark_web_populated
FROM credit_card_project.bronze.cards;


In [0]:
%sql
-- ============================================================
-- 4C. TRANSACTIONS: NULL CHECK
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(user_id) AS user_id_populated,
    COUNT(card_id) AS card_id_populated,
    COUNT(year) AS year_populated,
    COUNT(month) AS month_populated,
    COUNT(day) AS day_populated,
    COUNT(time) AS time_populated,
    COUNT(amount) AS amount_populated,
    COUNT(use_chip) AS use_chip_populated,
    COUNT(merchant_name) AS merchant_name_populated,
    COUNT(merchant_city) AS merchant_city_populated,
    COUNT(merchant_state) AS merchant_state_populated,
    COUNT(zip) AS zip_populated,
    COUNT(mcc) AS mcc_populated,
    COUNT(errors) AS errors_populated,
    COUNT(is_fraud) AS is_fraud_populated
FROM credit_card_project.bronze.transactions;


These results give us a useful distinction between required fields and legitimately optional fields (e.g. merchant_state_populated, zip_populated etc)

Check 4 — NULL validation: PASS for required fields:
All 24,386,900 transactions have values for:

- user_id
- card_id
- year
- month
- day
- time
- amount
- use_chip
- merchant_name
- merchant_city
- mcc
- is_fraud

There are NULLs in:

- merchant_state → 2,722,821
- zip → 2,878,135
- errors → 23,998,469

##5. Value and Range Validation

This section checks whether numeric and categorical values fall within reasonable business or domain ranges.

The goal is to identify invalid values without assuming that every unusual value is necessarily incorrect.

In [0]:
%sql
-- ============================================================
-- 5A. USERS: RANGE VALIDATION
-- ============================================================

SELECT
    MIN(current_age) AS min_age,
    MAX(current_age) AS max_age,
    MIN(retirement_age) AS min_retirement_age,
    MAX(retirement_age) AS max_retirement_age,
    MIN(birth_year) AS min_birth_year,
    MAX(birth_year) AS max_birth_year,
    MIN(birth_month) AS min_birth_month,
    MAX(birth_month) AS max_birth_month,
    MIN(fico_score) AS min_fico_score,
    MAX(fico_score) AS max_fico_score,
    MIN(num_credit_cards) AS min_num_credit_cards,
    MAX(num_credit_cards) AS max_num_credit_cards
FROM credit_card_project.bronze.users;


In [0]:
%sql
-- ============================================================
-- 5B. TRANSACTIONS: VALUE / RANGE VALIDATION
-- ============================================================

SELECT
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    MIN(month) AS min_month,
    MAX(month) AS max_month,
    MIN(day) AS min_day,
    MAX(day) AS max_day
FROM credit_card_project.bronze.transactions;


In [0]:
%sql
-- ============================================================
-- 5C. TRANSACTIONS: CATEGORICAL VALUES
-- ============================================================

SELECT
    use_chip,
    COUNT(*) AS transaction_count
FROM credit_card_project.bronze.transactions
GROUP BY use_chip
ORDER BY transaction_count DESC;


In [0]:
%sql
-- ============================================================
-- 5D. TRANSACTIONS: FRAUD VALUES
-- ============================================================

SELECT
    is_fraud,
    COUNT(*) AS transaction_count
FROM credit_card_project.bronze.transactions
GROUP BY is_fraud
ORDER BY transaction_count DESC;


In [0]:
%sql
-- ============================================================
-- 5E. TRANSACTIONS: AMOUNT FORMAT
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(
        TRY_CAST(REPLACE(amount, '$', '') AS DECIMAL(18,2))
    ) AS valid_amounts,
    COUNT(*) -
    COUNT(
        TRY_CAST(REPLACE(amount, '$', '') AS DECIMAL(18,2))
    ) AS invalid_amounts
FROM credit_card_project.bronze.transactions;


##6. Silver transformation validation

Here we validate what clean_transactions.sql actually accomplished.

We'll check:

- Silver row count vs Bronze
- Silver schema
- Amount conversion
- Date construction/conversion
- NULLs introduced by cleaning
- Whether the Silver transformation preserved referential integrity

That will give us a compact but genuinely useful data-quality notebook.

In [0]:
%sql
-- ============================================================
-- 6A. SILVER VS BRONZE ROW COUNT
-- ============================================================

SELECT
    b.bronze_count,
    s.silver_count,
    b.bronze_count - s.silver_count AS row_count_difference
FROM
    (
        SELECT COUNT(*) AS bronze_count
        FROM credit_card_project.bronze.transactions
    ) b
CROSS JOIN
    (
        SELECT COUNT(*) AS silver_count
        FROM credit_card_project.silver.cleaned_transactions
    ) s;


In [0]:
%sql
-- ============================================================
-- 6B. SILVER SCHEMA VALIDATION
-- ============================================================

DESCRIBE credit_card_project.silver.cleaned_transactions;


In [0]:
%sql
-- ============================================================
-- 6C. SILVER: AMOUNT TRANSFORMATION
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS amount_populated,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    SUM(CASE WHEN is_negative_amount THEN 1 ELSE 0 END) AS negative_amount_rows,
    SUM(CASE WHEN amount < 0 THEN 1 ELSE 0 END) AS actual_negative_amount_rows
FROM credit_card_project.silver.cleaned_transactions;


In [0]:
%sql
-- ============================================================
-- 6D. SILVER: AMOUNT CONVERSION CHECK
-- ============================================================

SELECT COUNT(*) AS invalid_or_missing_amounts
FROM credit_card_project.silver.cleaned_transactions
WHERE amount IS NULL;


In [0]:
%sql
-- ============================================================
-- 6E. SILVER: TIMESTAMP VALIDATION
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(transaction_timestamp) AS timestamp_populated,
    COUNT(*) - COUNT(transaction_timestamp) AS missing_timestamps,
    MIN(transaction_timestamp) AS earliest_transaction,
    MAX(transaction_timestamp) AS latest_transaction
FROM credit_card_project.silver.cleaned_transactions;


In [0]:
%sql
-- ============================================================
-- 6F. SILVER: BOOLEAN VALIDATION
-- ============================================================

SELECT
    is_fraud,
    COUNT(*) AS transaction_count
FROM credit_card_project.silver.cleaned_transactions
GROUP BY is_fraud
ORDER BY is_fraud;


In [0]:
%sql
-- ============================================================
-- 6G. SILVER: ONLINE TRANSACTION VALIDATION
-- ============================================================

SELECT
    is_online,
    COUNT(*) AS transaction_count
FROM credit_card_project.silver.cleaned_transactions
GROUP BY is_online
ORDER BY is_online;


In [0]:
%sql
-- ============================================================
-- 6H. SILVER: TRANSACTION → USER INTEGRITY
-- Expected result: 0
-- ============================================================

SELECT COUNT(*) AS orphaned_transactions
FROM credit_card_project.silver.cleaned_transactions t
LEFT JOIN credit_card_project.bronze.users u
    ON t.user_id = u.user_id
WHERE u.user_id IS NULL;


In [0]:
%sql
-- ============================================================
-- 6I. SILVER: TRANSACTION → CARD INTEGRITY
-- Expected result: 0
-- ============================================================

SELECT COUNT(*) AS orphaned_card_transactions
FROM credit_card_project.silver.cleaned_transactions t
LEFT JOIN credit_card_project.bronze.cards c
    ON t.user_id = c.user_id
   AND t.card_id = c.card_id
WHERE c.user_id IS NULL;


##7. Data Quality Findings

#Pipeline validation
- Bronze users: 2,000 rows
- Bronze cards: 6,146 rows
- Bronze transactions: 24,386,900 rows
- Silver cleaned transactions: 24,386,900 rows
- Bronze-to-Silver transaction row difference: 0

#Structural and integrity validation
- users.user_id is unique: PASS
- cards.user_id + card_id composite key: validated
- Orphaned cards: 0
- Transactions with unknown users: 0
- Transactions with unknown cards: 0
- Silver transactions with unknown users: validated
- Silver transactions with unknown cards: validated

#Source-data quality observations
- Required user and card fields contain no NULL values.
- Required transaction fields contain no NULL values.
- merchant_state and zip contain NULL values in the source data.
- errors is sparsely populated, consistent with an error/exception field.
- User age, retirement age, birth month, FICO score, and card-count ranges were reviewed and contained no immediately invalid values.
- Transaction dates and categorical values were reviewed for consistency.

#Duplicate transactions
The Bronze transaction dataset contains 66 exact duplicate rows, representing 33 duplicated transaction combinations.

These records were identified but not removed from Bronze. Bronze is intended to preserve the source data, while data cleaning and business rules are applied downstream.

#Silver transformation validation
- All transaction amounts were successfully converted to DECIMAL(18,2).
- No Silver transaction amounts are NULL.
- Negative amount flag matches the actual sign of the transaction amount.
- All transaction timestamps were successfully constructed.
- is_fraud was successfully converted to a Boolean field.
- is_online was successfully derived.
- Silver preserves the Bronze transaction row count.
#Overall conclusion

The Lakeflow Declarative Pipeline successfully ingested the source data into Bronze and transformed the transaction data into Silver without losing transaction records or breaking user/card relationships.

The primary source-data quality observation is the presence of 66 exact duplicate transaction rows. These duplicates are documented rather than removed from Bronze to preserve source-data fidelity.